In [ ]:
!pip install requests beautifulsoup4 openai langchain-openai

In [24]:
import requests
from bs4 import BeautifulSoup
from google.colab import userdata

url_article = userdata.get('ARTICLE')

def extract_text_from_url(url):
    response = requests.get(url)
    if response.status_code == 200:
      soup = BeautifulSoup(response.text, 'html.parser')
      for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()
      text = soup.get_text(separator= ' ')
      #Clean text
      lines = (line.strip() for line in text.splitlines())
      parts = (phrase.strip() for line in lines for phrase in line.split("  "))
      clean_text = '\n'.join(part for part in parts if part)
      return clean_text
    else:
      print(f"Failed to retrieve the URL. Status code: {response.status_code}")
      return None

extracted_text = extract_text_from_url(url_article)

In [26]:
from langchain_openai.chat_models.azure import AzureChatOpenAI

client = AzureChatOpenAI(
    azure_endpoint=userdata.get('AZURE_ENDPOINT'),
    api_key=userdata.get('API_KEY'),
    api_version="2024-02-15-preview",
    deployment_name="gpt-4o-mini",
    max_retries=0
)

def translate_article(text, lang):
  messages = [
      ("system", "Você atua como tradutor de textos"),
      ("user", f"Traduza o {text} para o idioma {lang} e responda em markdown")
  ]

  response = client.invoke(messages)
  return response.content

text = extract_text_from_url(url_article)
result = translate_article(text, "portuguese")
print(result)

# Azure OpenAI vs Azure AI Search: Qual a Diferença?

À medida que as ferramentas de IA se tornam mais amplamente adotadas, muitas equipes que utilizam o Microsoft Azure se deparam com a mesma pergunta: qual é a diferença entre Azure OpenAI e Azure AI Search? Ambos parecem poderosos. Ambos fazem parte dos serviços de IA do Azure, mas servem a propósitos muito diferentes.

Se você está construindo um aplicativo, chatbot ou recurso de pesquisa orientado por IA, entender o que cada ferramenta faz e quando usá-la pode economizar seu tempo e ajudá-lo a construir de maneira mais inteligente e rápida. Vamos detalhar.

## O que é o Azure OpenAI?

O Azure OpenAI Service oferece acesso aos modelos de linguagem da OpenAI, como GPT-4 e GPT-3.5, hospedados de forma segura no Azure. Esses modelos são projetados para entender e gerar texto semelhante ao humano, o que os torna perfeitos para tarefas como:

- Responder a perguntas em linguagem natural
- Escrever conteúdo ou resumos
- Traduzir textos
- 